In [ ]:
!git clone https://github.com/kanumi-2005/vehicle-counting
%cd vehicle-counting
!git submodule update --init --recursive

Cloning into 'vehicle-counting'...
remote: Enumerating objects: 243, done.
remote: Counting objects: 100% (243/243), done.
remote: Compressing objects: 100% (141/141), done.
remote: Total 243 (delta 111), reused 221 (delta 89), pack-reused 0 (from 0)
Receiving objects: 100% (243/243), 35.32 MiB | 43.00 MiB/s, done.
Resolving deltas: 100% (111/111), done.
/content/vehicle-counting
Submodule 'third_party/TrackEval' (https://github.com/kanumi-2005/TrackEval) registered for path 'third_party/TrackEval'
Cloning into '/content/vehicle-counting/third_party/TrackEval'...
Submodule path 'third_party/TrackEval': checked out '14511152b7e2b5b05efc631c0484a3536946382e'


In [ ]:
!pip install -q ultralytics kaggle tqdm pillow supervision trackers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.6/273.6 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 7.2 MB/s eta 0:00:00


In [ ]:
import json

kaggle_data = {
    "username": "username",
    "key": "key"
}

with open("kaggle.json", "w") as file:
    json.dump(kaggle_data, file, indent=4)

print("Đã tạo file kaggle.json")

Đã tạo file kaggle.json


In [ ]:
!kaggle datasets download -d bratjay/ua-detrac-orig -p /content/vehicle-counting/data/raw

Dataset URL: https://www.kaggle.com/datasets/bratjay/ua-detrac-orig
License(s): unknown
100% 9.23G/9.23G [01:25<00:00, 116MB/s]



In [ ]:
import zipfile
from pathlib import Path

zip_path = Path("/content/vehicle-counting/data/raw/ua-detrac-orig.zip")

extract_dir = Path("/content/vehicle-counting/data/raw/")

extract_dir.mkdir(parents=True, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

In [ ]:
!mv /content/vehicle-counting/data/raw/DETRAC-Images/DETRAC-Images/* /content/vehicle-counting/data/raw/DETRAC-Images/
!mv /content/vehicle-counting/data/raw/DETRAC-Train-Annotations-XML/DETRAC-Train-Annotations-XML/* /content/vehicle-counting/data/raw/DETRAC-Train-Annotations-XML/
!mv /content/vehicle-counting/data/raw/DETRAC-Test-Annotations-XML/DETRAC-Test-Annotations-XML/* /content/vehicle-counting/data/raw/DETRAC-Test-Annotations-XML/

In [ ]:
!./src/preprocessing.py --split val

[OK] Reusing split manifest: /content/vehicle-counting/data/preprocessed/splits.json

=== PREPROCESS VAL (12 SEQS) ===
[OK] Preprocessed: MVI_20012 (936 images)
[OK] Preprocessed: MVI_20051 (906 images)
[OK] Preprocessed: MVI_20052 (694 images)
[OK] Preprocessed: MVI_20061 (800 images)
[OK] Preprocessed: MVI_39771 (570 images)
[OK] Preprocessed: MVI_39781 (1865 images)
[OK] Preprocessed: MVI_39811 (1070 images)
[OK] Preprocessed: MVI_40732 (2120 images)
[OK] Preprocessed: MVI_40871 (1720 images)
[OK] Preprocessed: MVI_40991 (1820 images)
[OK] Preprocessed: MVI_63521 (2055 images)
[OK] Preprocessed: MVI_63554 (1445 images)


In [ ]:
!./src/convert_to_trackeval_mot.py --split val


=== CONVERT TRACKEVAL VAL (12 SEQS) ===
[OK] TrackEval val: MVI_20012 (seqLength=936, annotated_frames=936)
[OK] TrackEval val: MVI_20051 (seqLength=906, annotated_frames=906)
[OK] TrackEval val: MVI_20052 (seqLength=694, annotated_frames=694)
[OK] TrackEval val: MVI_20061 (seqLength=800, annotated_frames=800)
[OK] TrackEval val: MVI_39771 (seqLength=570, annotated_frames=570)
[OK] TrackEval val: MVI_39781 (seqLength=1865, annotated_frames=1861)
[OK] TrackEval val: MVI_39811 (seqLength=1070, annotated_frames=500)
[OK] TrackEval val: MVI_40732 (seqLength=2120, annotated_frames=2120)
[OK] TrackEval val: MVI_40871 (seqLength=1720, annotated_frames=1720)
[OK] TrackEval val: MVI_40991 (seqLength=1820, annotated_frames=1667)
[OK] TrackEval val: MVI_63521 (seqLength=2055, annotated_frames=2055)
[OK] TrackEval val: MVI_63554 (seqLength=1445, annotated_frames=1445)
[OK] Seqmap saved: /content/vehicle-counting/data/trackeval/data/gt/seqmaps/ua-detrac-val.txt


In [ ]:
import os

os.environ["MODELPATH"] = "/content/drive/MyDrive/ua_detrac_runs_ignored_box_new/yolov8n_detrac/weights/best.pt"

In [ ]:
!./src/run_and_export_trackeval_mot.py \
  --split val \
  --model-path "$MODELPATH" \
  --tracker sort \
  --confidence-threshold 0.2 \
  --track-activation-threshold 0.494 \
  # --high-conf-detection-threshold 0.494

[INFO] Tracker output name: yolo-sort
[INFO] Tracking config: {'tracker_type': 'sort', 'lost_track_buffer': 75, 'track_activation_threshold': 0.494, 'minimum_consecutive_frames': 2, 'minimum_iou_threshold': 0.1, 'high_conf_detection_threshold': 0.6}
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

=== RUN TRACKING VAL (12 SEQS) ===
[OK] val | MVI_20012 -> 8596 lines
[OK] val | MVI_20051 -> 10121 lines
[OK] val | MVI_20052 -> 9411 lines
[OK] val | MVI_20061 -> 9386 lines
[OK] val | MVI_39771 -> 2903 lines
[OK] val | MVI_39781 -> 9805 lines
[OK] val | MVI_39811 -> 900 lines
[OK] val | MVI_40732 -> 12493 lines
[OK] val | MVI_40871 -> 33906 lines
[OK] val | MVI_40991 -> 4642 lines
[OK] val | MVI_63521 -> 15001 lines
[OK] val | MVI_

In [ ]:
!./src/run_tracking_eval.bash \
  --split val \
  --tracker sort

STARTING EVALUATION
------------------------------------------
  Benchmark     : ua-detrac
  Detector      : from config
  Tracker       : sort
  Tracker Name  : yolo-sort
  Split         : val
  GT Folder     : /content/vehicle-counting/data/trackeval/data/gt
  Tracker Folder: /content/vehicle-counting/data/trackeval/data/trackers
  Metrics       : HOTA CLEAR Identity

CLEAR Config:
METRICS              : ['HOTA', 'CLEAR', 'Identity'] 
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          

Identity Config:
METRICS              : ['HOTA', 'CLEAR', 'Identity'] 
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          

Evaluating 1 tracker(s) on 12 sequence(s) for 1 class(es) on MotChallenge2DBox dataset using the following metrics: HOTA, CLEAR, Identity, Count


Evaluating yolo-sort

    MotChallenge2DBox.get_raw_seq_data(yolo-sort, MVI_20012)               0.1549 sec
    MotChalleng

In [ ]:
!./src/run_and_export_trackeval_mot.py \
  --split val \
  --model-path "$MODELPATH" \
  --tracker byte \
  --confidence-threshold 0.2 \
  --track-activation-threshold 0.494 \
  --high-conf-detection-threshold 0.494

[INFO] Tracker output name: yolo-byte
[INFO] Tracking config: {'tracker_type': 'byte', 'lost_track_buffer': 75, 'track_activation_threshold': 0.494, 'minimum_consecutive_frames': 2, 'minimum_iou_threshold': 0.1, 'high_conf_detection_threshold': 0.494}

=== RUN TRACKING VAL (12 SEQS) ===
[OK] val | MVI_20012 -> 8405 lines
[OK] val | MVI_20051 -> 9691 lines
[OK] val | MVI_20052 -> 9277 lines
[OK] val | MVI_20061 -> 9156 lines
[OK] val | MVI_39771 -> 2875 lines
[OK] val | MVI_39781 -> 9716 lines
[OK] val | MVI_39811 -> 899 lines
[OK] val | MVI_40732 -> 12457 lines
[OK] val | MVI_40871 -> 33255 lines
[OK] val | MVI_40991 -> 4606 lines
[OK] val | MVI_63521 -> 14701 lines
[OK] val | MVI_63554 -> 11250 lines

[DONE] Finished running tracking.


In [ ]:
!./src/run_tracking_eval.bash \
  --split val \
  --tracker byte

STARTING EVALUATION
------------------------------------------
  Benchmark     : ua-detrac
  Detector      : from config
  Tracker       : byte
  Tracker Name  : yolo-byte
  Split         : val
  GT Folder     : /content/vehicle-counting/data/trackeval/data/gt
  Tracker Folder: /content/vehicle-counting/data/trackeval/data/trackers
  Metrics       : HOTA CLEAR Identity

CLEAR Config:
METRICS              : ['HOTA', 'CLEAR', 'Identity'] 
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          

Identity Config:
METRICS              : ['HOTA', 'CLEAR', 'Identity'] 
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          

Evaluating 1 tracker(s) on 12 sequence(s) for 1 class(es) on MotChallenge2DBox dataset using the following metrics: HOTA, CLEAR, Identity, Count


Evaluating yolo-byte

    MotChallenge2DBox.get_raw_seq_data(yolo-byte, MVI_20012)               0.2660 sec
    MotChalleng

In [ ]:
!./src/preprocessing.py --split test

[OK] Reusing split manifest: /content/vehicle-counting/data/preprocessed/splits.json

=== PREPROCESS TEST (40 SEQS) ===
[OK] Preprocessed: MVI_39031 (1470 images)
[OK] Preprocessed: MVI_39051 (1120 images)
[OK] Preprocessed: MVI_39211 (1660 images)
[OK] Preprocessed: MVI_39271 (1570 images)
[OK] Preprocessed: MVI_39311 (1505 images)
[OK] Preprocessed: MVI_39361 (2030 images)
[OK] Preprocessed: MVI_39371 (1390 images)
[OK] Preprocessed: MVI_39401 (1385 images)
[OK] Preprocessed: MVI_39501 (540 images)
[OK] Preprocessed: MVI_39511 (380 images)
[OK] Preprocessed: MVI_40701 (1130 images)
[OK] Preprocessed: MVI_40711 (1030 images)
[OK] Preprocessed: MVI_40712 (2400 images)
[OK] Preprocessed: MVI_40714 (1180 images)
[OK] Preprocessed: MVI_40742 (1655 images)
[OK] Preprocessed: MVI_40743 (1630 images)
[OK] Preprocessed: MVI_40761 (2030 images)
[OK] Preprocessed: MVI_40762 (1825 images)
[OK] Preprocessed: MVI_40763 (1745 images)
[OK] Preprocessed: MVI_40771 (1720 images)
[OK] Preprocessed: MVI

In [ ]:
!./src/convert_to_trackeval_mot.py --split test


=== CONVERT TRACKEVAL TEST (40 SEQS) ===
[OK] TrackEval test: MVI_39031 (seqLength=1470, annotated_frames=1470)
[OK] TrackEval test: MVI_39051 (seqLength=1120, annotated_frames=1051)
[OK] TrackEval test: MVI_39211 (seqLength=1660, annotated_frames=1566)
[OK] TrackEval test: MVI_39271 (seqLength=1570, annotated_frames=1566)
[OK] TrackEval test: MVI_39311 (seqLength=1505, annotated_frames=1505)
[OK] TrackEval test: MVI_39361 (seqLength=2030, annotated_frames=2030)
[OK] TrackEval test: MVI_39371 (seqLength=1390, annotated_frames=1390)
[OK] TrackEval test: MVI_39401 (seqLength=1385, annotated_frames=1385)
[OK] TrackEval test: MVI_39501 (seqLength=540, annotated_frames=540)
[OK] TrackEval test: MVI_39511 (seqLength=380, annotated_frames=380)
[OK] TrackEval test: MVI_40701 (seqLength=1130, annotated_frames=1130)
[OK] TrackEval test: MVI_40711 (seqLength=1030, annotated_frames=1030)
[OK] TrackEval test: MVI_40712 (seqLength=2400, annotated_frames=2400)
[OK] TrackEval test: MVI_40714 (seqLeng

In [ ]:
!./src/run_and_export_trackeval_mot.py \
  --split test \
  --model-path "$MODELPATH" \
  --tracker byte \
  --confidence-threshold 0.2 \
  --track-activation-threshold 0.494 \
  --high-conf-detection-threshold 0.494

[INFO] Tracker output name: yolo-byte
[INFO] Tracking config: {'tracker_type': 'byte', 'lost_track_buffer': 75, 'track_activation_threshold': 0.494, 'minimum_consecutive_frames': 2, 'minimum_iou_threshold': 0.1, 'high_conf_detection_threshold': 0.494}

=== RUN TRACKING TEST (40 SEQS) ===
[OK] test | MVI_39031 -> 6409 lines
[OK] test | MVI_39051 -> 3140 lines
[OK] test | MVI_39211 -> 3997 lines
[OK] test | MVI_39271 -> 9784 lines
[OK] test | MVI_39311 -> 21885 lines
[OK] test | MVI_39361 -> 11269 lines
[OK] test | MVI_39371 -> 6690 lines
[OK] test | MVI_39401 -> 15324 lines
[OK] test | MVI_39501 -> 5105 lines
[OK] test | MVI_39511 -> 2268 lines
[OK] test | MVI_40701 -> 15073 lines
[OK] test | MVI_40711 -> 8486 lines
[OK] test | MVI_40712 -> 25912 lines
[OK] test | MVI_40714 -> 43546 lines
[OK] test | MVI_40742 -> 26542 lines
[OK] test | MVI_40743 -> 13701 lines
[OK] test | MVI_40761 -> 16101 lines
[OK] test | MVI_40762 -> 12110 lines
[OK] test | MVI_40763 -> 7869 lines
[OK] test | MVI_4

In [ ]:
!./src/run_tracking_eval.bash \
  --split test \
  --tracker byte

STARTING EVALUATION
------------------------------------------
  Benchmark     : ua-detrac
  Detector      : from config
  Tracker       : byte
  Tracker Name  : yolo-byte
  Split         : test
  GT Folder     : /content/vehicle-counting/data/trackeval/data/gt
  Tracker Folder: /content/vehicle-counting/data/trackeval/data/trackers
  Metrics       : HOTA CLEAR Identity

CLEAR Config:
METRICS              : ['HOTA', 'CLEAR', 'Identity'] 
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          

Identity Config:
METRICS              : ['HOTA', 'CLEAR', 'Identity'] 
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          

Evaluating 1 tracker(s) on 40 sequence(s) for 1 class(es) on MotChallenge2DBox dataset using the following metrics: HOTA, CLEAR, Identity, Count


Evaluating yolo-byte

    MotChallenge2DBox.get_raw_seq_data(yolo-byte, MVI_39031)               0.3550 sec
    MotChallen

In [ ]:
!cp -r /content/vehicle-counting/data/trackeval/ /content/drive/MyDrive/tracking-results/new/